# Week 5: Visualization and Communication

**Foundations of Data Science**

## In this lesson

1. A chart is an argument: start from the sentence, not the data
2. Choosing the form: five questions, five chart types
3. The same data, four ways: watching a chart get better
4. Uncertainty on the page: error bars, bands, and small multiples
5. Things that lie: truncated axes, dual axes, pie charts, 3D, rainbow colormaps
6. Audiences: the plot for you, the plot for the clinic, the plot for the paper

## The one idea

**Every chart should be the picture of one sentence.** Write the sentence first ("diabetes prevalence rises steeply after 40, and faster for people who are obese"), then build the chart that makes a reader see it in under five seconds. If you can't write the sentence, you don't know what the chart is for yet, and you're still doing EDA (last week), not communication (this week).

## Setup

<!-- nb-links -->
[Download this notebook](week_05_visualization.ipynb?download=1){download="week_05_visualization.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/notebooks/week_05/week_05_visualization.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/week_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("week_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this week's raw data only if it isn't already on disk, then build the processed table
for component in ["nhanes_demo", "nhanes_bmx", "nhanes_bpx", "nhanes_diq"]:
    if not Path(f"data/raw/{component}.parquet").exists():
        !python data_gen.py fetch --name {component}
if not Path("data/processed/nhanes.parquet").exists():
    !python preprocessing.py

print("Working directory:", os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

nhanes = pd.read_parquet("data/processed/nhanes.parquet")
adults = nhanes[nhanes["age"] >= 20].copy()
adults["sex"] = adults["sex"].map({1: "male", 2: "female"})
adults["race_eth"] = adults["race_eth"].map({1: "Mexican American", 2: "Other Hispanic", 3: "White", 4: "Black", 6: "Asian", 7: "Other/multi"})
adults["age_band"] = pd.cut(adults["age"], bins=[20, 30, 40, 50, 60, 70, 81], labels=["20-29", "30-39", "40-49", "50-59", "60-69", "70+"], right=False)
adults["bmi_cat"] = pd.cut(adults["bmi"], bins=[0, 25, 30, 100], labels=["normal", "overweight", "obese"], right=False)

# One helper used throughout: prevalence with a 95% CI for any grouping
def prevalence_ci(df: pd.DataFrame, by: list[str], col: str = "diabetes") -> pd.DataFrame:
    g = df.groupby(by, observed=True)[col].agg(["mean", "count"]).reset_index()
    se = np.sqrt(g["mean"] * (1 - g["mean"]) / g["count"])
    g["lo"], g["hi"] = g["mean"] - 1.96 * se, g["mean"] + 1.96 * se
    return g

## 1. The sentence comes first

Last week's EDA produced a findings log. Each row is a candidate sentence. For the clinic, the sentence we've been building toward since week 1:

> *Diabetes is rare in normal-weight adults under 50, and common in obese adults over 50; the screening rule should depend on both.*

Everything below is about turning that sentence into a picture a medical director would accept in a two-minute meeting.

## 2. Choosing the form

Five questions you can ask of data, and the chart that answers each:

| Question | Chart | Example today |
|---|---|---|
| How is one thing distributed? | histogram, density, box | BMI |
| How does a quantity compare across groups? | bar (dot plot if many groups) | prevalence by race/ethnicity |
| How does a quantity change along an ordered axis? | line | prevalence by age band |
| How do two quantities relate? | scatter | BMI vs waist |
| How does a comparison change across a third variable? | small multiples, or lines colored by group | prevalence by age, one line per BMI category |

Two rules that cover most mistakes. **Position beats everything**: people read differences in position (along a shared axis) far more accurately than differences in length, angle, area, or color. **Order the categories**: alphabetical order is never the right order for a bar chart; sort by value, or by the natural order of the variable.

## 3. The same data, four ways

Prevalence by age band and BMI category. Version 1 is what you get by default. Each version after it fixes one thing.

In [ ]:
prev = prevalence_ci(adults.dropna(subset=["bmi_cat"]), by=["age_band", "bmi_cat"])

# Version 1: the default. Grouped bars, default colors, no title, no uncertainty.
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=prev, x="age_band", y="mean", hue="bmi_cat", ax=ax)
plt.show()

Problems: the reader has to compare bar heights across gaps; the legend says `bmi_cat` and the y-axis says `mean`, neither of which is a word a clinician uses; and there's no sense of how sure we are. It answers "what are the numbers" rather than making the sentence visible.

**Version 2: lines, because age is ordered.** A line connects the age bands and shows the *shape* of the rise. Labels become words.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.lineplot(data=prev, x="age_band", y="mean", hue="bmi_cat", marker="o", ax=ax)
ax.set_xlabel("Age")
ax.set_ylabel("Share with diabetes")
ax.legend(title="BMI category")
ax.set_title("Diabetes prevalence by age and weight")
plt.show()

**Version 3: show the uncertainty.** The 20-29 obese cell has about 350 people and 1.4% prevalence; the interval matters. Error bars or a band around each line keep readers from over-reading small differences. Also: percent format on the axis, and a gridline a reader can use.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for cat, g in prev.groupby("bmi_cat", observed=True):
    ax.errorbar(g["age_band"].astype(str), g["mean"], yerr=[g["mean"] - g["lo"], g["hi"] - g["mean"]],
                marker="o", capsize=3, label=cat)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("Age")
ax.set_ylabel("Share with diabetes (95% CI)")
ax.legend(title="BMI category")
ax.set_title("Diabetes prevalence by age and weight")
ax.grid(axis="y", alpha=0.3)
plt.show()

**Version 4: make the sentence the title, and point at the evidence.** A title that states the finding does more work than any other element on the page. One annotation marks the thing the clinic is deciding about. Colors become meaningful (an ordered palette for an ordered category) and the legend sits where the lines end so the eye doesn't have to travel.

In [ ]:
palette = {"normal": "#4C72B0", "overweight": "#DD8452", "obese": "#C44E52"}
fig, ax = plt.subplots(figsize=(9, 5))
for cat, g in prev.groupby("bmi_cat", observed=True):
    x = g["age_band"].astype(str)
    ax.plot(x, g["mean"], marker="o", color=palette[cat], linewidth=2, label=cat)
    ax.fill_between(x, g["lo"], g["hi"], color=palette[cat], alpha=0.15)
    ax.text(x.iloc[-1], g["mean"].iloc[-1], f"  {cat}", color=palette[cat], va="center", fontweight="bold")

ax.axvspan(-0.5, 1.5, color="gray", alpha=0.08)
ax.text(0.5, 0.36, "under 40: screening\nis the open question", ha="center", va="top", fontsize=9, color="gray")

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_ylim(0, 0.42)
ax.set_xlabel("Age")
ax.set_ylabel("Share reporting diabetes")
ax.set_title("Diabetes is rare in normal-weight adults under 50 and common in obese adults over 50",
             loc="left", fontsize=12, fontweight="bold", pad=12)
fig.text(0.01, -0.02, "Source: NHANES 2017-2018, adults 20+, unweighted. Shaded bands are 95% confidence intervals.", fontsize=8, color="gray")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.show()

That's the chart that goes in front of the medical director. Compare it to version 1: same numbers, and a reader gets the sentence without being told it.

## 4. Uncertainty on the page

Three ways to show it, in order of preference:

1. **Bands or error bars** on the estimate itself (above). Best when there are few series.
2. **Small multiples**: one small panel per group, same axes everywhere, so the eye compares positions across panels. Best when a single panel would have more than three or four lines.
3. **Showing the data** behind the summary (a strip or swarm plot under a box plot) when n is small enough to see individual points.

Small multiples are the under-used one. Here's race/ethnicity by age, which has six groups and would be unreadable as six lines on one axis.

In [ ]:
prev_race = prevalence_ci(adults, by=["race_eth", "age_band"])
order = adults.groupby("race_eth")["diabetes"].mean().sort_values(ascending=False).index  # sort panels by crude rate

g = sns.FacetGrid(prev_race, col="race_eth", col_wrap=3, col_order=order, height=3, aspect=1.2, sharey=True)
g.map_dataframe(lambda data, color, **kw: plt.errorbar(
    data["age_band"].astype(str), data["mean"], yerr=[data["mean"] - data["lo"], data["hi"] - data["mean"]],
    marker="o", capsize=2, color="#C44E52"))
g.set_titles("{col_name}")
g.set_axis_labels("Age", "Share with diabetes")
for ax in g.axes.flat:
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.grid(axis="y", alpha=0.3)
g.fig.suptitle("Age drives prevalence in every group; the groups differ in level, not in shape", y=1.03, x=0.01, ha="left", fontweight="bold")
plt.show()

## 5. Things that lie

Each of these is common, and each misleads a reader who is trusting you.

- **Truncated y-axis on a bar chart.** Bars encode length from zero; start them at 20% and a 2-point difference looks like a doubling. Lines can start anywhere (they encode position), bars can't.
- **Dual y-axes.** Two scales on one chart let you make any two series look correlated by choosing the scales. Use two panels.
- **Pie charts.** Angles are hard to compare. If you must show parts of a whole, a sorted bar chart does it better; a stacked bar does it if the whole is also changing.
- **3D anything.** Perspective distorts every comparison and adds nothing.
- **Rainbow colormaps** (`jet`). They have artificial bright bands that create edges in the data that aren't there, and they're unreadable for the 8% of men with red-green color blindness. Use `viridis` for sequential, `RdBu` for diverging, and a qualitative palette with at most six colors for categories.
- **Unlabeled, un-sourced, un-dated.** A chart that leaves the notebook needs its data source, population, date, and n on it somewhere. Version 4 has that line in gray under the chart.

A demonstration of the first one, because it's the most common.

In [ ]:
crude = adults.groupby("race_eth")["diabetes"].mean().sort_values()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (ymin, title) in zip(axes, [(0.13, "axis starts at 13%: looks like a 3x gap"), (0, "axis starts at 0: the honest version")]):
    ax.bar(crude.index, crude.values, color="#4C72B0")
    ax.set_ylim(ymin, 0.19)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_title(title, loc="left")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 6. Three audiences, three plots

The same finding is drawn differently depending on who's reading.

| Audience | What they need | What changes |
|---|---|---|
| **You, during analysis** | speed, many views | defaults are fine; `eda_toolkit` one-liners; no titles |
| **The decision-maker** | one sentence, one chart, no jargon | finding-as-title, annotation on the decision point, plain words, CI bands, source line |
| **A paper or report** | precision, reproducibility | neutral title, full axis labels with units, n per group in a caption or table, vector format, colorblind-safe palette, every number in the figure also in a table |

Last week's `outcome_crosstab_plot` and `box_violin_plot` calls are audience-one plots: fast, many at once, for your eyes. Version 4 above is audience two. For audience three you would drop the annotation, state "prevalence by age band and BMI category" as the title, and put the n per cell in the caption.

One habit that pays off: **save every outward-facing figure from code**, never by screenshot. `fig.savefig("figures/prevalence_age_bmi.png", dpi=200, bbox_inches="tight")` and the SVG twin for the paper. When the data change, the figure regenerates; when a reviewer asks for n, it's in the script.

In [ ]:
from pathlib import Path

# Save the decision-maker chart from code. figures/ is gitignored like data/; the script that makes it is what's versioned.
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/week05_axis_truncation.png", dpi=200, bbox_inches="tight")
print("saved figures/week05_axis_truncation.png")

## Looking ahead
- **Project**: proposal pitches are in week 6, five minutes each. One chart, one sentence, your plan.
- **Reading**: ISLR chapter 3 (linear regression), sections 3.1 and 3.2. Next week we fit the first model.